importing Required modules

In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.cross_decomposition import PLSRegression


In [ ]:
from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)


df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

In [ ]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

In [ ]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

In [ ]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

In [ ]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

In [ ]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [ ]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [ ]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [ ]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


In [ ]:
encodermyth=Sequential()
encodermyth.add(Dense(units=8000,activation="relu"))
encodermyth.add(Dense(units=4000,activation="relu"))
encodermyth.add(Dense(units=2000,activation="relu"))
encodermyth.add(Dense(units=1000,activation="relu"))
encodermyth.add(Dense(units=500,activation="relu"))
encodermyth.add(Dense(units=280,activation="relu"))

decodermyth=Sequential()
decodermyth.add(Dense(units=500,activation="relu"))
decodermyth.add(Dense(units=1000,activation="relu"))
decodermyth.add(Dense(units=2000,activation="relu"))
decodermyth.add(Dense(units=4000,activation="relu"))
decodermyth.add(Dense(units=8000,activation="relu"))
decodermyth.add(Dense(units=15330,activation="softmax"))
autoencodermyth = Sequential([encodermyth,decodermyth])
autoencodermyth.compile(loss="mse",optimizer=SGD(lr=1.5))
#standardize the data

scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
#PCA
# fit pca on data

autoencodermyth.fit(X1,X1,batch_size = 16, shuffle =True, epochs=5)


Z1 = encodermyth.predict(X1)




In [ ]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

In [ ]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(168).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

In [ ]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(168).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(168)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

In [ ]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=168)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

In [ ]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [ ]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


Applying PCA dimensioality reduction technique

In [ ]:
#PCA
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
#fit pca on data
encodermrna=Sequential()
encodermrna.add(Dense(units=12000,activation="relu"))
encodermrna.add(Dense(units=6000,activation="relu"))
encodermrna.add(Dense(units=3000,activation="relu"))
encodermrna.add(Dense(units=1500,activation="relu"))
encodermrna.add(Dense(units=750,activation="relu"))
encodermrna.add(Dense(units=300,activation="relu"))

decodermrna=Sequential()
decodermrna.add(Dense(units=750,activation="relu"))
decodermrna.add(Dense(units=1500,activation="relu"))
decodermrna.add(Dense(units=3000,activation="relu"))
decodermrna.add(Dense(units=6000,activation="relu"))
decodermrna.add(Dense(units=12000,activation="relu"))
decodermrna.add(Dense(units=12723,activation="relu"))
autoencodermrna = Sequential([encodermrna,decodermrna])
autoencodermrna.compile(loss="mse",optimizer=SGD(lr=1.5))
autoencodermrna.fit(X3,X3,batch_size = 16, shuffle =True, epochs=4)
Z3 = encodermrna.predict(X3)


Applying RFE Feature selection methods

In [ ]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(180)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

In [ ]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=180)
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

In [ ]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


In [ ]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

In [ ]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
encodercna=Sequential()
encodercna.add(Dense(units=12000,activation="relu"))
encodercna.add(Dense(units=6000,activation="relu"))
encodercna.add(Dense(units=3000,activation="relu"))
encodercna.add(Dense(units=1500,activation="relu"))
encodercna.add(Dense(units=750,activation="relu"))
encodercna.add(Dense(units=500,activation="relu"))
encodercna.add(Dense(units=300,activation="relu"))

decodercna=Sequential()
decodercna.add(Dense(units=500,activation="relu"))
decodercna.add(Dense(units=750,activation="relu"))
decodercna.add(Dense(units=1500,activation="relu"))
decodercna.add(Dense(units=3000,activation="relu"))
decodercna.add(Dense(units=6000,activation="relu"))
decodercna.add(Dense(units=12000,activation="relu"))
decodercna.add(Dense(units=23311,activation="relu"))
autoencodercna = Sequential([encodercna,decodercna])
autoencodercna.compile(loss="mse",optimizer=SGD(lr=1.5))
autoencodercna.fit(X4,X4,epochs=5,batch_size = 16, shuffle =True)
Z4 = encodercna.predict(X4)

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

In [ ]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

In [ ]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


In [ ]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=180)
rfe.fit(Z4, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(180)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Merging dataset

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)






In [ ]:
epochs = 30
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

In [ ]:
import deepsurvk

dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  epochs=epochs,
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)


In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 10
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)



In [ ]:
result = np.zeros((5, 4, 5))

result[0][0][0]=c_index_test
result[0][0][1]=mse
result[0][0][2]=rmse
result[0][0][3]=mae
result[0][0][4]=mdae

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][0][0]=c_index
result[1][0][1]=mse
result[1][0][2]=r2
result[1][0][3]=mae
result[1][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][0][0]=c_index
result[2][0][1]=mse
result[2][0][2]=r2
result[2][0][3]=mae
result[2][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][0][0]=c_index
result[3][0][1]=mse
result[3][0][2]=r2
result[3][0][3]=mae
result[3][0][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][0][0]=c_index
result[4][0][1]=mse
result[4][0][2]=r2
result[4][0][3]=mae
result[4][0][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  epochs=epochs,
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][1][0]=c_index_test
result[0][1][1]=mse
result[0][1][2]=rmse
result[0][1][3]=mae
result[0][1][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][1][0]=c_index
result[1][1][1]=mse
result[1][1][2]=r2
result[1][1][3]=mae
result[1][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][1][0]=c_index
result[2][1][1]=mse
result[2][1][2]=r2
result[2][1][3]=mae
result[2][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][1][0]=c_index
result[3][1][1]=mse
result[3][1][2]=r2
result[3][1][3]=mae
result[3][1][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][1][0]=c_index
result[4][1][1]=mse
result[4][1][2]=r2
result[4][1][3]=mae
result[4][1][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  epochs=epochs,
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][2][0]=c_index_test
result[0][2][1]=mse
result[0][2][2]=rmse
result[0][2][3]=mae
result[0][2][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][2][0]=c_index
result[1][2][1]=mse
result[1][2][2]=r2
result[1][2][3]=mae
result[1][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][2][0]=c_index
result[2][2][1]=mse
result[2][2][2]=r2
result[2][2][3]=mae
result[2][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][2][0]=c_index
result[3][2][1]=mse
result[3][2][2]=r2
result[3][2][3]=mae
result[3][2][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][2][0]=c_index
result[4][2][1]=mse
result[4][2][2]=r2
result[4][2][3]=mae
result[4][2][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train,
                  batch_size=n_patients_train,
                  epochs=epochs,
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][3][0]=c_index_test
result[0][3][1]=mse
result[0][3][2]=rmse
result[0][3][3]=mae
result[0][3][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][3][0]=c_index
result[1][3][1]=mse
result[1][3][2]=r2
result[1][3][3]=mae
result[1][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][3][0]=c_index
result[2][3][1]=mse
result[2][3][2]=r2
result[2][3][3]=mae
result[2][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][3][0]=c_index
result[3][3][1]=mse
result[3][3][2]=r2
result[3][3][3]=mae
result[3][3][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][3][0]=c_index
result[4][3][1]=mse
result[4][3][2]=r2
result[4][3][3]=mae
result[4][3][4]=mdae

In [ ]:
!pip install pandas openpyxl
data_0_0 = result[0, 0:4, 0]
data_1_0 = result[3, 0:4, 0]
data_2_0 = result[2, 0:4, 0]
data_3_0 = result[1, 0:4, 0]
data_4_0 = result[4, 0:4, 0]
combined_data_0 = np.concatenate((data_0_0, data_1_0, data_2_0, data_3_0, data_4_0))
data_0_1 = result[0, 0:4, 1]
data_1_1 = result[3, 0:4, 1]q
data_2_1 = result[2, 0:4, 1]
data_3_1 = result[1, 0:4, 1]
data_4_1 = result[4, 0:4, 1]
combined_data_1 = np.concatenate((data_0_1, data_1_1, data_2_1, data_3_1, data_4_1))
data_0_2 = result[0, 0:4, 2]
data_1_2 = result[3, 0:4, 2]
data_2_2 = result[2, 0:4, 2]
data_3_2 = result[1, 0:4, 2]
data_4_2 = result[4, 0:4, 2]
combined_data_2 = np.concatenate((data_0_2, data_1_2, data_2_2, data_3_2, data_4_2))
data_0_3 = result[0, 0:4, 3]
data_1_3 = result[3, 0:4, 3]
data_2_3 = result[2, 0:4, 3]
data_3_3 = result[1, 0:4, 3]
data_4_3 = result[4, 0:4, 3]
combined_data_3 = np.concatenate((data_0_3, data_1_3, data_2_3, data_3_3, data_4_3))
data_0_4 = result[0, 0:4, 4]
data_1_4 = result[3, 0:4, 4]
data_2_4 = result[2, 0:4, 4]
data_3_4 = result[1, 0:4, 4]
data_4_4 = result[4, 0:4, 4]
combined_data_4 = np.concatenate((data_0_4, data_1_4, data_2_4, data_3_4, data_4_4))
# Create a DataFrame
df = pd.DataFrame({
    'Combined_Data_0': combined_data_0,
    'Combined_Data_1': combined_data_1,
    'Combined_Data_2': combined_data_2,
    'Combined_Data_3': combined_data_3,
    'Combined_Data_4': combined_data_4
})

# Save the DataFrame to an Excel file
df.to_excel('combined_data.xlsx', index=False)

print("Data has been saved to combined_data.xlsx")